# Four imported CSL methods: graph-recovery sanity check

Run this prerequisite before the larger transition studies. Conventional c-GC, c-GC*, fastC-GC and fastC-GC* receive identical four-neuron recordings. A directed chain has links 0→1 and 1→2 (the second is negative); neuron 3 is isolated. The null has no links. Truth is used only for scoring. Native outputs become unsigned support, without a fitted VAR readout.
The linear lagged data generator supplies known circuit truth; it is not a fitted VAR method. The four estimators share one family, and empty-graph agreement is uninformative. Successful graph recovery remains separate from transition-driver validation.
Install the official conventional package using `../tools/conventional-cgc/README.md`. This notebook uses the main project kernel; its adapter invokes the pinned CPU environment.
Run All fits the complete three-seed, four-method sanity panel by default. Results use a separate resumable run-label directory. Set `RUN_SANITY=False` only when you intend to inspect saved evidence instead of fitting.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


In [ ]:
import json
from dataclasses import asdict

import pandas as pd
from IPython.display import Image, Markdown, display

from effectome.experiments.csl_sanity import CSLSanityConfig, run_csl_sanity
from effectome.experiments.csl_sanity_reporting import render_csl_sanity_report
from effectome.workflows.cross_method import load_cross_method_profile
from notebooks._shared import PROJECT_ROOT, require_conventional_runtime

RUN_SANITY = True  # Run All fits the complete native sanity panel.
RUN_LABEL = "faster-machine-02"  # Reuse unchanged settings to resume; change after protocol/code edits.
config = CSLSanityConfig(seeds=(0, 1, 2), n_samples=1024)
_, methods = load_cross_method_profile("csl_sanity")
OUTPUT_DIRECTORY = PROJECT_ROOT / "outputs" / "csl-sanity" / "reruns" / RUN_LABEL
FROZEN_DIRECTORY = PROJECT_ROOT / "synthetic" / "results" / "four_csl_sanity_2026-11-15"
print(asdict(config))
print("Imported method panel:", list(methods))
print("Planned fits:", 2 * len(config.seeds) * len(methods))
print("Fitting enabled:", RUN_SANITY, "| outputs:", OUTPUT_DIRECTORY)
print("Conventional CPU permutation seeds are not fully controlled upstream.")


## Recovery and nontrivial agreement

All independent data seeds enter the summary. The first declared seed illustrates the recovered circuit. Compare truth, recall and false edges before interpreting agreement; methods can agree on mistakes. No discovery thresholds are selected using these results.


In [ ]:
report = None
evidence_directory = OUTPUT_DIRECTORY
path = OUTPUT_DIRECTORY / "sanity.json"
if RUN_SANITY:
    require_conventional_runtime(methods)
    report = run_csl_sanity(config, OUTPUT_DIRECTORY, methods=methods)
    render_csl_sanity_report(report, OUTPUT_DIRECTORY / "analysis")
elif path.is_file():
    report = json.loads(path.read_text())
elif (FROZEN_DIRECTORY / "sanity.json").is_file():
    evidence_directory = FROZEN_DIRECTORY
    report = json.loads((FROZEN_DIRECTORY / "sanity.json").read_text())

if report is not None:
    print("Saved configuration:", report["config"])
    if report["config"] != json.loads(json.dumps(asdict(config))):
        print("Saved results use a different configuration; enable RUN_SANITY to fit the requested one.")
    print("Evidence:", evidence_directory)
    print("Available / attempted:", report["available_runs"], "/", report["attempted_runs"])
    display(pd.DataFrame(report["seed_rows"]).drop(columns="support", errors="ignore"))
    display(pd.DataFrame(report["pairwise_rows"]))
else:
    print("No saved study. Inspect the method settings, then enable RUN_SANITY.")


In [ ]:
for name in ("analysis-report.md", "stats-appendix.md", "figure-catalog.md"):
    path = evidence_directory / "analysis" / name
    if path.is_file():
        display(Markdown(path.read_text()))
for name in ("figure-01-native-support.png", "figure-02-seed-recovery.png"):
    path = evidence_directory / "analysis" / "figures" / name
    if path.is_file():
        display(Image(filename=str(path)))

if report is not None:
    for method, result in report["diagnostics"].items():
        print(method, result["status"])
        if result["status"] != "available":
            print(result["reason"])
print("Next: 02 shared transitions; 04 framework validation; 05 driver recovery; 06 mechanism revision.")
